In [ ]:
%run main.ipynb

In [ ]:
#Baseline Modal
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# 先分出 test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=26
)
#X_train_val → 80% 的特征，X_test → 20% 的特征，y_train_val → 80% 的标签，y_test → 20% 的标签

# 从X_train_val, y_train_val里再分 validation（分成X_train, X_val, y_train, y_val）
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1, random_state=26
)#这里的0.1是80%里的10%


# 在标准化前，先保存 raw（未标准化）版本：后面kNN需要用原始经纬度
X_train_raw = X_train.copy()
X_val_raw   = X_val.copy()
X_test_raw  = X_test.copy()

# scale (fit only on train)#标准化（只在训练集）并转回DataFrame
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns,index=X_train_raw.index)
X_val   = pd.DataFrame(scaler.transform(X_val), columns=X.columns,index=X_val_raw.index)
X_test  = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test_raw.index)
#scaler.transform 标准化 test 数据
#index=X_test_raw.index Pandas 会默认生成新的 0~n 索引
#加了 index=...，这样索引不会乱（对齐 y / 后面 debug 更稳）

# model (no internal early stopping, since you already have X_val)
model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=300,
    early_stopping=False,
    random_state=26
)

model.fit(X_train, y_train)
#测试集和预测集的预测
val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

# 训练集预测
train_pred = model.predict(X_train)
# 计算训练误差
train_mse = mean_squared_error(y_train, train_pred)

print("Baseline Train MSE:", train_mse)#加入是为了判断是不是欠拟合或者过拟合
print("Validation MSE:", mean_squared_error(y_val, val_pred))#Mean Squared Error（均方误差）
print("Test MSE:", mean_squared_error(y_test, test_pred))

In [ ]:
#检测训练集中是不是有出现多个样本经纬度完全一样
print("numbers of Duplicate coordinates：",X_train_raw[['Latitude','Longitude']].duplicated().sum())

duplicates = X_train_raw[X_train_raw[['Latitude','Longitude']].duplicated(keep=False)]

print(duplicates[['Latitude','Longitude']].sort_values(['Latitude','Longitude']))

X_train_raw.groupby(['Latitude','Longitude']).size().sort_values(ascending=False).head()
#训练集中确实有大量“经纬度完全相同”的样本
#对某个点来说，不止一个距离=0 的邻居，这会让 idxs = idxs[1:] 这种“删第一个当作自己”的逻辑不够严谨。
#只删掉第一个 idxs[1:]：可能删掉了“同坐标的另一个样本”,但仍然可能保留了自己的那条样本（如果它不是排在第一个）
#也可能保留多个 distance=0 的点（包含自己的 y），有轻微标签泄漏风险
#不能使用下列代码
'''
ArithmeticErrordef neighbour_mean_price(query_df, y_train_series, exclude_self=False):
    """
    为 query_df 的每个样本计算：训练集中k个邻居的 y 的平均值
    query_df: 需要计算邻居特征的数据（train/val/test 的 raw DataFrame）
    y_train_series: 训练集标签（只能用它，避免data leakage）
    exclude_self: 如果 query_df 是训练集本身，要剔除“自己”这个邻居
    """
    # indices：每个query点在训练集中对应的k个邻居的索引
    _, indices = knn.kneighbors(query_df[['Latitude', 'Longitude']])

    neigh_mean = []  # 存放每个样本的邻居均价
    for idxs in indices:
        if exclude_self:
            # train 查询时，第一个邻居通常是自己，要去掉
            idxs = idxs[1:K+1]
        else:
            # val/test 直接取前K个邻居（它们全来自train）
            idxs = idxs[:K]
        # 用训练集的 y_train 取邻居价格并求平均（关键：不使用val/test真实y）
        neigh_mean.append(y_train_series.iloc[idxs].mean())

    return np.array(neigh_mean)
'''

# 1. 统计每个坐标出现多少次
coord_counts = X_train_raw.groupby(['Latitude', 'Longitude']).size()

# 2. 统计“出现次数”的分布
distribution = coord_counts.value_counts().sort_index()

# 3. 转成表格并加列名
distribution_df = distribution.reset_index()
distribution_df.columns = ['Repetition frequency of coordinate points', 'number of coordinate points']

# 4. 打印标题和表格
print("\n=== Repetitive statistics based on latitude and longitude ===")
print(distribution_df)

#用knn的局限性
#因为有非常多的坐标重复，所以会导致加入邻居特征时，在删除自己之后选择k个邻居选择到的领居出现不一样的情况。
#这种情况发生在和删除掉原坐标完全相同的坐标的情况非常少，在K值选择7的时候，有3个坐标，32个重复样本
#sklearn 在 多个距离相同的邻居之间没有唯一排序，返回的邻居集合可能因为内部实现顺序不同而变化
#用random seeds不能避免这种情况，random_state 只控制“随机过程”，只影响train_test_split，MLPRegressor 的权重初始化和可能的内部随机优化过程
#这种影响预测率的情况不是一定会发生，只是可能存在这种风险
#虽然代码每次运行其实返回的是同一组邻居（因为数据顺序不变，sklearn版本不变，算法参数不变），但sklearn实际上在距离相同的情况下就按内部顺序返回前 K 个
#有可能内部顺序改变一下可能预测准确率会提高或下降，所以我们找到的可能不是最优的参数解

'''
| 方法                 | 问题                |
| ------------------ | ----------------- |
| kNN                | 邻居数量固定，但邻居组合可能不唯一 |
| distance threshold | 邻居组合唯一，但邻居数量不固定   |
'''

In [ ]:
# 检查训练集中每个点的第一个邻居是否就是自己（通常为True）
d, ind = knn.kneighbors(X_train_raw[['Latitude','Longitude']])
print("Self in first neighbor:", np.mean(ind[:,0] == np.arange(len(X_train_raw))))

In [ ]:
#判断 训练集每个样本的 kneighbors 返回结果里到底有没有包含自己
import numpy as np

# 用你当前的 knn（已 fit 在 X_train_raw 上），以及当前 K
# 注意：这里检查的是训练集自身 query
dists, inds = knn.kneighbors(X_train_raw[['Latitude', 'Longitude']])

n = len(X_train_raw)

# 对训练集来说，“自己”的训练位置索引就是 0..n-1
self_pos = np.arange(n)

# A: 自己不在邻居列表里
self_in_list = (inds == self_pos[:, None]).any(axis=1)
count_A = np.sum(~self_in_list)

# B: 自己在邻居列表里，但不一定第一个
count_B = np.sum(self_in_list)

# B1: 自己在列表里且是第一个邻居
count_B_first = np.sum(inds[:, 0] == self_pos)

# B2: 自己在列表里但不是第一个
count_B_not_first = np.sum(self_in_list & (inds[:, 0] != self_pos))

print(f"Total train samples: {n}")
print(f"Case A (self NOT in returned neighbors): {count_A} ({count_A/n:.3%})")
print(f"Case B (self IN returned neighbors):     {count_B} ({count_B/n:.3%})")
print(f"  - B1: self is the 1st neighbor:        {count_B_first} ({count_B_first/n:.3%})")
print(f"  - B2: self NOT the 1st neighbor:       {count_B_not_first} ({count_B_not_first/n:.3%})")

# 给几个例子看看（最多打印5个）
if count_A > 0:
    ex = np.where(~self_in_list)[0][:5]
    print("\nExamples of Case A (row_i where self missing):", ex.tolist())
    print("Their neighbor indices (first row shown):")
    print(inds[ex[0]])
    print("Distances of that row:")
    print(dists[ex[0]])

if count_B_not_first > 0:
    ex = np.where(self_in_list & (inds[:, 0] != self_pos))[0][:5]
    print("\nExamples of Case B2 (self present but not first):", ex.tolist())
    r = ex[0]
    pos = np.where(inds[r] == r)[0][0]
    print(f"For row {r}, self appears at position {pos} in neighbor list.")
    print("Neighbor indices:", inds[r])
    print("Distances:", dists[r])